# utils

In [1]:
import json

import os
import numpy as np
from tqdm.auto import tqdm

import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
import matplotlib.pyplot as plt

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    #check image_file is a path or image
    if isinstance(image_file, str):
        image = Image.open(image_file).convert('RGB')
    else:
        image = image_file
    image = image.resize((input_size, input_size))

    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

# inference

In [2]:
!ls /teamspace/studios/uit-fine-tuning/work_dirs/internvl_chat_v2_0

Vintern_1B_v2_finetune_lora_data34
Vintern_1B_v2_finetune_lora_data34_merge
Vintern_1B_v2_finetune_lora_data43
Vintern_1B_v2_finetune_lora_data43_merge
Vintern_1B_v2_finetune_lora_data_balanced_v1
Vintern_1B_v2_finetune_lora_data_balanced_v1_merge
Vintern_1B_v2_finetune_lora_upto4k
Vintern_1B_v2_finetune_lora_upto4k_merge


In [10]:
!ls /teamspace/studios/this_studio/work_dirs/internvl_chat_v2_0/


Vintern_1B_v2_finetune_lora_data34_merge
Vintern_1B_v2_finetune_lora_data_23
Vintern_1B_v2_finetune_lora_data_23_merge
Vintern_1B_v2_finetune_lora_data_balanced_v2
Vintern_1B_v2_finetune_lora_data_balanced_v2_merge


In [11]:
# 0.4403
# model_name = '/teamspace/studios/uit-fine-tuning/work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_data34_merge'

# ?
model_name = '/teamspace/studios/this_studio/work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_data_23_merge'
model = AutoModel.from_pretrained(model_name,
                                  torch_dtype=torch.bfloat16,
                                  low_cpu_mem_usage=True,
                                  trust_remote_code=True).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [12]:
# test_path = '/teamspace/studios/this_studio/dev-test/vimmsd-public-test.json'
# 
# 
test_path = '/teamspace/studios/uit-fine-tuning/private-test-images/vimmsd-private-test.json'

preds = []
debug = False
with open(test_path, 'r') as f:
    data = json.load(f)
    i = 0
    for sample in tqdm(data.items()):
        if debug: print(sample)
        sample_id = sample[0]
        # sample_image_path = '/teamspace/studios/this_studio/dev-test/dev-images/'+sample[1]['image'] 

        sample_image_path = '/teamspace/studios/uit-fine-tuning/private-test-images/test-images/'+sample[1]['image'] 

        sample_image_caption = sample[1]['caption']        
        if debug:
            image = Image.open(sample_image_path).convert('RGB')
            plt.figure(figsize=(5,5))
            plt.imshow(image)
            plt.show()
        
        ### inference 
        pixel_values = load_image(sample_image_path, max_num=12).to(torch.bfloat16).cuda()
        generation_config = dict(max_new_tokens= 2048, do_sample=False, 
                                 num_beams = 3, repetition_penalty=2.0,
                                 pad_token_id=tokenizer.eos_token_id)
        
        question = f"""<image>
            Đi kèm bình luận: {sample_image_caption}.
            Bối cảnh: [social media platform].
            Hãy phân tích sắc thái mải mai của cả nội dung trong ảnh và nội dung bình luận. 
            Sau đó xác định nhãn cho nội dung trên, chỉ lựa chọn một trong bốn nhãn sau: 
            'châm biếm qua hình ảnh', 'châm biếm cả hình ảnh và văn bản', 'không châm biếm', 'châm biếm qua văn bản'.
            Sắc thái:"""
        ##
        response = model.chat(tokenizer, pixel_values, question, generation_config)
        
        ###--
        if debug:
            print(f'User: {question}\nAssistant: {response}')
            print("="*30)
        ###--
        
        # update preds
        preds.append({
            'id': sample_id,
            'label' : response
        })
        i += 1
        if debug and i == 2: break

if not debug:
    # saving
    with open('my_preds.json', 'w') as f:
        json.dump(preds, f)

  0%|          | 0/1504 [00:00<?, ?it/s]

# submit

In [15]:
import json
import pandas as pd

data = json.load(open('/teamspace/studios/this_studio/my_preds.json'))
df = pd.DataFrame(data)
df['label'].value_counts()


label
châm biếm cả hình ảnh và văn bản    924
không châm biếm                     444
châm biếm qua văn bản                97
châm biếm qua hình ảnh               39
Name: count, dtype: int64

In [16]:
df.to_csv(f"pred_{model_name.split('/')[-1]}.csv")

In [17]:
label_mapping ={
            'image-sarcasm':'châm biếm qua hình ảnh', 
            'multi-sarcasm':'châm biếm cả hình ảnh và văn bản', 
            'not-sarcasm':'không châm biếm', 
            'text-sarcasm': 'châm biếm qua văn bản'
        }
inv_map = {v: k for k, v in label_mapping.items()}

phase = "test"  # or whatever phase you want to specify
# Convert preds list to dictionary with 'id' as the key and 'label' as the value
# results_dict = {pred['id']: idx_to_classes[pred['label']] for pred in preds}
results_dict = {pred['id']: inv_map[pred['label']] for pred in data}

# Create the final structure
output = {
    "results": results_dict,
    "phase": phase
}

print(len(output['results']))


with open(f"results_{model_name.split('/')[-1]}.json", 'w') as out_file:
     json.dump(output, out_file)
print('done')

1504
done
